# Phase 1: PDF -> Page Images (Multi-Modal RAG)

In [ ]:
# !pip install -q pymupdf pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 28.0 MB/s eta 0:00:00


In [1]:
import os
import shutil
import fitz  # PyMuPDF
import json
import logging
from pathlib import Path
from dataclasses import dataclass, asdict
from PIL import Image
from tqdm import tqdm

In [2]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("pdf_to_images")

### Config

In [3]:
@dataclass(frozen=True)
class RenderConfig:
    dpi: int = 150
    max_side_px: int = 1536
    image_format: str = "PNG"


SOURCES = {
    "paper": {
        "pdf_dir": "raw/papers",
        "output_dir": "extracted/Page Images/papers",
        "done_dir": "raw/papers_done"
    },
    "book": {
        "pdf_dir": "raw/books",
        "output_dir": "extracted/Page Images/books",
        "done_dir": "raw/books_done"
    },
}

PAGE_METADATA_FILE = Path("extracted/Page Images/metadata/page_metadata.jsonl")
RENDER_CFG = RenderConfig()

### Core rendering logic

In [4]:
def resize_if_needed(image: Image.Image, max_side_px: int) -> Image.Image:
    """Downscale an image so its longest side <= max_side_px, preserving aspect ratio."""
    width, height = image.size
    longest_side = max(width, height)

    if longest_side <= max_side_px:
        return image

    scale = max_side_px / longest_side
    new_size = (int(width * scale), int(height * scale))
    return image.resize(new_size, Image.LANCZOS)

In [5]:
def render_pdf_to_images(pdf_path: Path, output_dir: Path, cfg: RenderConfig) -> list[dict]:
    """Render every page of a PDF to a PNG file. Returns per-page metadata records."""
    output_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    zoom = cfg.dpi / 72  # PyMuPDF's default page unit is 72 DPI
    matrix = fitz.Matrix(zoom, zoom)

    page_records = []

    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        pix = page.get_pixmap(matrix=matrix)

        image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        image = resize_if_needed(image, cfg.max_side_px)

        page_number = page_index + 1  # 1-indexed for human readability
        image_filename = f"page_{page_number:04d}.png"
        image_path = output_dir / image_filename

        image.save(image_path, format=cfg.image_format)

        page_records.append({
            "page_number": page_number,
            "image_path": str(image_path),
            "width": image.width,
            "height": image.height,
        })

    doc.close()
    return page_records

In [6]:
def is_already_processed(output_dir: Path, expected_page_count: int) -> bool:
    """Resume support: skip documents whose images are already fully rendered."""
    if not output_dir.exists():
        return False
    existing_pages = list(output_dir.glob("page_*.png"))
    return len(existing_pages) == expected_page_count

In [7]:
def get_page_count(pdf_path: Path) -> int:
    with fitz.open(pdf_path) as doc:
        return len(doc)

### Corpus-level driver

In [8]:
def process_corpus(doc_type: str, pdf_dir: Path, output_dir: Path, done_dir: Path, cfg: RenderConfig) -> list[dict]:
    if not pdf_dir.exists():
        logger.warning(f"[{doc_type}] Source directory not found: {pdf_dir}")
        return []

    # Ensure the "done" directory exists
    done_dir.mkdir(parents=True, exist_ok=True)

    pdf_files = sorted(list({p for p in pdf_dir.glob("*") if p.suffix.lower() == ".pdf"}))
    logger.info(f"[{doc_type}] Found {len(pdf_files)} PDFs in {pdf_dir}")

    all_records = []
    succeeded, skipped, failed = 0, 0, 0

    for pdf_path in tqdm(pdf_files, desc=f"Rendering {doc_type}s"):
        doc_id = pdf_path.stem
        doc_output_dir = output_dir / doc_id

        try:
            expected_pages = get_page_count(pdf_path)

            if is_already_processed(doc_output_dir, expected_pages):
                skipped += 1
                page_records = [
                    {
                        "page_number": int(p.stem.split("_")[1]),
                        "image_path": str(p),
                        "width": None,
                        "height": None,
                    }
                    for p in sorted(doc_output_dir.glob("page_*.png"))
                ]
                
                # Move already processed/cached files to the done directory as well
                shutil.move(str(pdf_path), str(done_dir / pdf_path.name))
            else:
                page_records = render_pdf_to_images(pdf_path, doc_output_dir, cfg)
                succeeded += 1
                
                # Move newly processed file to the done directory
                shutil.move(str(pdf_path), str(done_dir / pdf_path.name))

            for record in page_records:
                record.update({
                    "doc_id": doc_id,
                    "doc_type": doc_type,
                    "total_pages": len(page_records),
                    "source_pdf": str(done_dir / pdf_path.name), # update metadata to point to the new location
                })
                all_records.append(record)

        except Exception as e:
            logger.error(f"[{doc_type}] Failed on {pdf_path.name}: {e}")
            failed += 1
            continue

    logger.info(f"[{doc_type}] Done. Rendered: {succeeded} | Skipped & Moved: {skipped} | Failed: {failed}")
    return all_records

In [9]:
def write_metadata(records: list[dict], output_file: Path) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    logger.info(f"Wrote {len(records)} page records to {output_file}")

### Run

In [11]:
if __name__ == "__main__":
    all_page_records = []

    for doc_type, paths in SOURCES.items():
        records = process_corpus(
            doc_type=doc_type,
            pdf_dir=Path(paths["pdf_dir"]),
            output_dir=Path(paths["output_dir"]),
            done_dir=Path(paths["done_dir"]),
            cfg=RENDER_CFG,
        )
        all_page_records.extend(records)

    write_metadata(all_page_records, PAGE_METADATA_FILE)

    print(f"\n✅ Total pages rendered across corpus: {len(all_page_records)}")

2026-07-01 14:40:45,601 | INFO | [paper] Found 360 PDFs in raw\papers
Rendering papers:  90%|████████▉ | 323/360 [25:06<03:08,  5.08s/it]  

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find ExtGState resource 'a0'

MuPDF error: syntax error: cannot find E

Rendering papers: 100%|██████████| 360/360 [26:45<00:00,  4.46s/it]
2026-07-01 15:07:31,164 | INFO | [paper] Done. Rendered: 360 | Skipped & Moved: 0 | Failed: 1
2026-07-01 15:07:31,165 | INFO | [book] Found 20 PDFs in raw\books
Rendering books: 100%|██████████| 20/20 [16:51<00:00, 50.58s/it]
2026-07-01 15:24:22,824 | INFO | [book] Done. Rendered: 20 | Skipped & Moved: 0 | Failed: 0
2026-07-01 15:24:22,919 | INFO | Wrote 22589 page records to extracted\Page Images\metadata\page_metadata.jsonl



✅ Total pages rendered across corpus: 22589


### Validation

In [15]:
import os
import fitz  # PyMuPDF
from pathlib import Path
from PIL import Image

# Define paths matching your local setup
SOURCES = {
    "paper": {
        "pdf_dir": Path("raw/papers_done"),  # Checking the files that finished
        "output_dir": Path("extracted/Page Images/papers"),
    },
    "book": {
        "pdf_dir": Path("raw/books_done"),
        "output_dir": Path("extracted/Page Images/books"),
    },
}


In [16]:
def validate_extraction():
    all_passed = True

    for doc_type, paths in SOURCES.items():
        pdf_dir = paths["pdf_dir"]
        output_dir = paths["output_dir"]

        if not pdf_dir.exists():
            print(f"ℹ️ [{doc_type.upper()}] No finished directory found at {pdf_dir}. Skipping.")
            continue

        pdf_files = sorted(pdf_dir.glob("*.pdf")) + sorted(pdf_dir.glob("*.PDF"))
        # Handle Windows double-counting just in case
        pdf_files = sorted(list({p for p in pdf_files if p.suffix.lower() == ".pdf"}))

        print(f"\n🔍 Verifying {len(pdf_files)} {doc_type.upper()}s...")
        
        passed_count = 0
        failed_docs = []

        for pdf_path in pdf_files:
            doc_id = pdf_path.stem
            doc_output_dir = output_dir / doc_id
            
            # 1. Check if the extraction folder exists
            if not doc_output_dir.exists():
                print(f"❌ FAILED: Folder missing for '{pdf_path.name}'")
                failed_docs.append((pdf_path.name, "Missing target directory"))
                all_passed = False
                continue

            try:
                # 2. Get true expected page count
                with fitz.open(pdf_path) as doc:
                    expected_pages = len(doc)

                # 3. Get actual generated image count
                found_images = sorted(doc_output_dir.glob("page_*.png"))
                
                if len(found_images) != expected_pages:
                    print(f"❌ FAILED: '{pdf_path.name}' page count mismatch! Expected {expected_pages}, found {len(found_images)}.")
                    failed_docs.append((pdf_path.name, f"Page mismatch (Expected {expected_pages}, got {len(found_images)})"))
                    all_passed = False
                    continue

                # 4. Deep check: Verify images are uncorrupted and open cleanly
                corrupted_images = 0
                for img_path in found_images:
                    # Check file size
                    if img_path.stat().st_size == 0:
                        corrupted_images += 1
                        continue
                    
                    # Try opening the visual data structure
                    try:
                        with Image.open(img_path) as img:
                            img.verify()  # Verifies the file integrity without loading all pixels into memory
                    except Exception:
                        corrupted_images += 1

                if corrupted_images > 0:
                    print(f"❌ FAILED: '{pdf_path.name}' has {corrupted_images} broken/empty image files.")
                    failed_docs.append((pdf_path.name, f"{corrupted_images} corrupted image files"))
                    all_passed = False
                    continue

                # If it makes it here, everything is clean
                passed_count += 1

            except Exception as e:
                print(f"💥 ERROR processing validation for '{pdf_path.name}': {e}")
                failed_docs.append((pdf_path.name, f"Validation processing error: {e}"))
                all_passed = False

        # Summary for this type
        print(f"📊 Status [{doc_type.upper()}]: {passed_count}/{len(pdf_files)} passed validation.")
        if failed_docs:
            print(f"   ⚠️ The following documents need reprocessing:")
            for name, reason in failed_docs:
                print(f"   - {name} ➔ Reason: {reason}")

    if all_passed:
        print("\n🏆 SUCCESS: All extracted page images are perfect, uncorrupted, and complete!")
    else:
        print("\n⚠️ ATTENTION: Some documents failed validation checks. Review the log items above.")

In [17]:
if __name__ == "__main__":
    validate_extraction()


🔍 Verifying 360 PAPERs...
📊 Status [PAPER]: 360/360 passed validation.

🔍 Verifying 20 BOOKs...
📊 Status [BOOK]: 20/20 passed validation.

🏆 SUCCESS: All extracted page images are perfect, uncorrupted, and complete!
